# Part I DDP Benchmark Notebook (VFI + PFI)

This notebook runs **discrete dynamic programming (DDP)** benchmarks on the same Part-1 model setup as `01_part1_training.ipynb`:
- Basic model: VFI and PFI
- Risky debt model: outer equilibrium loop with inner VFI and inner PFI

It saves DDP checkpoints and compares baseline policies against available NN checkpoints (LR/ER/BR for basic, BR for risky).

In [ ]:
# =============================================================================
# 1. Imports and Setup
# =============================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys
import json
import dataclasses
from datetime import datetime
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from src.economy.parameters import EconomicParams
from src.economy.data import load_dataset_bundle
from src.ddp import DDPGridConfig, BasicModelDDP, RiskyModelDDP, save_ddp_solution
from src.utils.checkpointing import load_training_result


In [ ]:
# =============================================================================
# 2. Run Paths
# =============================================================================
NN_RESULTS_DIR = Path('..') / 'results' / 'latest'
NN_CHECKPOINTS_DIR = NN_RESULTS_DIR / 'checkpoints'

DDP_RUN_NAME = datetime.now().strftime('run-ddp-%Y%m%d-%H%M%S')
DDP_RESULTS_DIR = Path('..') / 'results' / DDP_RUN_NAME
DDP_FIGURES_DIR = DDP_RESULTS_DIR / 'figures'
DDP_CHECKPOINTS_DIR = DDP_RESULTS_DIR / 'checkpoints'

for d in [DDP_RESULTS_DIR, DDP_FIGURES_DIR, DDP_CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'NN checkpoints: {NN_CHECKPOINTS_DIR}')
print(f'DDP results dir: {DDP_RESULTS_DIR}')
print(f'DDP checkpoints: {DDP_CHECKPOINTS_DIR}')


In [ ]:
# =============================================================================
# 3. Benchmark Parameters (Aligned with 01_part1_training.ipynb)
# =============================================================================
base_params = EconomicParams()

BASIC_SCENARIOS = {
    'baseline': {
        'description': 'No adjustment costs',
        'params': base_params.with_overrides(
            cost_convex=0.0,
            cost_fixed=0.0,
            cost_inject_fixed=0.0,
            cost_inject_linear=0.0,
        ),
    },
    'smooth_cost': {
        'description': 'Convex adjustment costs only',
        'params': base_params.with_overrides(
            cost_convex=0.2,
            cost_fixed=0.0,
            cost_inject_fixed=0.0,
            cost_inject_linear=0.0,
        ),
    },
    'fixed_cost': {
        'description': 'Fixed adjustment costs only',
        'params': base_params.with_overrides(
            cost_convex=0.0,
            cost_fixed=0.02,
            cost_inject_fixed=0.0,
            cost_inject_linear=0.0,
        ),
    },
}

RISKY_SCENARIOS = {
    'baseline': {
        'description': 'Standard risky debt with all cost components',
        'params': base_params.with_overrides(
            cost_convex=0.01,
            cost_fixed=0.01,
            cost_inject_fixed=0.02,
            cost_inject_linear=0.02,
            cost_default=0.4,
            tax=0.3,
        ),
    }
}

print('Basic scenarios:', list(BASIC_SCENARIOS.keys()))
print('Risky scenarios:', list(RISKY_SCENARIOS.keys()))


In [ ]:
# =============================================================================
# 4. Dataset Bundle + Metadata Bounds (Single Source of Truth)
# =============================================================================
# Optional override:
# export DDP_DATASET_PATH=/absolute/path/to/train_xxxxx.npz

def _discover_dataset_path() -> Path:
    env_path = os.environ.get('DDP_DATASET_PATH')
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f'DDP_DATASET_PATH not found: {p}')
        return p

    candidate_dirs = [
        Path('..') / 'data',
        Path('.') / 'data',
        NN_RESULTS_DIR / 'data',
    ]

    candidates = []
    for d in candidate_dirs:
        if d.exists():
            candidates.extend(d.glob('train_*.npz'))
            candidates.extend(d.glob('train*.npz'))

    if not candidates:
        raise FileNotFoundError(
            'No training dataset .npz found. Set DDP_DATASET_PATH or place train_*.npz under ../data.'
        )

    # Use the newest file by modified time
    candidates = sorted(candidates, key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0]

DATASET_PATH = _discover_dataset_path()
train_bundle = load_dataset_bundle(str(DATASET_PATH), require_metadata=True)

required_keys = {'z', 'z_next_main'}
missing_keys = required_keys - set(train_bundle.data.keys())
if missing_keys:
    raise ValueError(f'Dataset missing required keys for offline DDP transitions: {sorted(missing_keys)}')

k_bounds = tuple(train_bundle.bounds['k'])
logz_bounds = tuple(train_bundle.bounds['log_z'])
b_bounds = tuple(train_bundle.bounds['b'])

print(f'Using dataset: {DATASET_PATH}')
print(f'Dataset fingerprint: {train_bundle.fingerprint}')
print(f'k_bounds   = {k_bounds}')
print(f'logz_bounds= {logz_bounds}')
print(f'b_bounds   = {b_bounds}')


In [ ]:
# =============================================================================
# 5. DDP Grid Profile
# =============================================================================
DDP_PROFILE = 'FAST_DEBUG'  # Options: FAST_DEBUG, BALANCED, PRODUCTION

if DDP_PROFILE == 'FAST_DEBUG':
    ddp_z_size = 7
    ddp_k_size = 40
    ddp_b_size = 25
elif DDP_PROFILE == 'BALANCED':
    ddp_z_size = 11
    ddp_k_size = 60
    ddp_b_size = 35
else:  # PRODUCTION
    ddp_z_size = 15
    ddp_k_size = 100
    ddp_b_size = 60

grid_config_basic = DDPGridConfig(
    z_size=ddp_z_size,
    k_size=ddp_k_size,
    b_size=2,  # not used by basic solver; must satisfy config validation
    grid_type='multiplicative',
)

grid_config_risky = DDPGridConfig(
    z_size=ddp_z_size,
    k_size=ddp_k_size,
    b_size=ddp_b_size,
    grid_type='multiplicative',
)

print('DDP profile:', DDP_PROFILE)
print('Basic grid:', dataclasses.asdict(grid_config_basic))
print('Risky grid:', dataclasses.asdict(grid_config_risky))


In [ ]:
# =============================================================================
# 6. Solve Basic Model with VFI and PFI (Offline Dataset-Driven)
# =============================================================================
ddp_basic_results = {}

for scenario_name, scenario in BASIC_SCENARIOS.items():
    params = scenario['params']
    model = BasicModelDDP.from_dataset_bundle(
        params=params,
        bundle=train_bundle,
        grid_config=grid_config_basic,
    )

    print(f'\n[Basic/{scenario_name}] Solving VFI...')
    v_vfi, policy_k_vfi = model.solve_basic_vfi(tol=1e-6, max_iter=2000)

    print(f'[Basic/{scenario_name}] Solving PFI...')
    v_pfi, policy_k_pfi = model.solve_basic_pfi(max_iter=200, eval_steps=400)

    value_gap = float(np.max(np.abs(v_vfi.numpy() - v_pfi.numpy())))
    policy_gap = float(np.max(np.abs(policy_k_vfi.numpy() - policy_k_pfi.numpy())))

    ddp_basic_results[scenario_name] = {
        'params': params,
        'model': model,
        'vfi': {'value': v_vfi, 'policy_k': policy_k_vfi},
        'pfi': {'value': v_pfi, 'policy_k': policy_k_pfi},
        'gaps': {'value_max_abs': value_gap, 'policy_k_max_abs': policy_gap},
    }

    save_ddp_solution(
        save_dir=str(DDP_CHECKPOINTS_DIR),
        model_name='basic',
        scenario_name=scenario_name,
        solver_name='vfi',
        value=v_vfi,
        policy_k=policy_k_vfi,
        params=params,
        shock_params=None,
        grid_config=grid_config_basic,
        metrics={
            'value_pfi_gap_max_abs': value_gap,
            'policy_k_pfi_gap_max_abs': policy_gap,
            'dataset_fingerprint': train_bundle.fingerprint,
        },
        overwrite=True,
        verbose=False,
    )

    save_ddp_solution(
        save_dir=str(DDP_CHECKPOINTS_DIR),
        model_name='basic',
        scenario_name=scenario_name,
        solver_name='pfi',
        value=v_pfi,
        policy_k=policy_k_pfi,
        params=params,
        shock_params=None,
        grid_config=grid_config_basic,
        metrics={
            'value_vfi_gap_max_abs': value_gap,
            'policy_k_vfi_gap_max_abs': policy_gap,
            'dataset_fingerprint': train_bundle.fingerprint,
        },
        overwrite=True,
        verbose=False,
    )

    print(f'  value max|VFI-PFI|={value_gap:.3e}, policy max|VFI-PFI|={policy_gap:.3e}')


In [ ]:
# =============================================================================
# 7. Solve Risky Debt Model with Inner VFI and Inner PFI (Offline Dataset-Driven)
# =============================================================================
ddp_risky_results = {}

for scenario_name, scenario in RISKY_SCENARIOS.items():
    params = scenario['params']
    model = RiskyModelDDP.from_dataset_bundle(
        params=params,
        bundle=train_bundle,
        grid_config=grid_config_risky,
    )

    print(f'\n[Risky/{scenario_name}] Solving outer loop + inner VFI...')
    v_vfi, (pk_vfi, pb_vfi), q_vfi = model.solve_risky_vfi(
        tol=1e-4,
        max_iter=20,
        damping_weight=0.7,
        inner_vfi_tol=1e-5,
        inner_vfi_max_iter=1200,
    )

    print(f'[Risky/{scenario_name}] Solving outer loop + inner PFI...')
    v_pfi, (pk_pfi, pb_pfi), q_pfi = model.solve_risky_pfi(
        tol=1e-4,
        max_iter=20,
        damping_weight=0.7,
        inner_pfi_max_iter=120,
        inner_pfi_eval_steps=300,
    )

    value_gap = float(np.max(np.abs(v_vfi.numpy() - v_pfi.numpy())))
    k_gap = float(np.max(np.abs(pk_vfi.numpy() - pk_pfi.numpy())))
    b_gap = float(np.max(np.abs(pb_vfi.numpy() - pb_pfi.numpy())))
    q_gap = float(np.max(np.abs(q_vfi.numpy() - q_pfi.numpy())))

    ddp_risky_results[scenario_name] = {
        'params': params,
        'model': model,
        'vfi': {'value': v_vfi, 'policy_k': pk_vfi, 'policy_b': pb_vfi, 'q': q_vfi},
        'pfi': {'value': v_pfi, 'policy_k': pk_pfi, 'policy_b': pb_pfi, 'q': q_pfi},
        'gaps': {
            'value_max_abs': value_gap,
            'policy_k_max_abs': k_gap,
            'policy_b_max_abs': b_gap,
            'q_max_abs': q_gap,
        },
    }

    save_ddp_solution(
        save_dir=str(DDP_CHECKPOINTS_DIR),
        model_name='risky',
        scenario_name=scenario_name,
        solver_name='vfi',
        value=v_vfi,
        policy_k=pk_vfi,
        policy_b=pb_vfi,
        bond_price=q_vfi,
        params=params,
        shock_params=None,
        grid_config=grid_config_risky,
        metrics={
            'value_pfi_gap_max_abs': value_gap,
            'policy_k_pfi_gap_max_abs': k_gap,
            'policy_b_pfi_gap_max_abs': b_gap,
            'q_pfi_gap_max_abs': q_gap,
            'dataset_fingerprint': train_bundle.fingerprint,
        },
        overwrite=True,
        verbose=False,
    )

    save_ddp_solution(
        save_dir=str(DDP_CHECKPOINTS_DIR),
        model_name='risky',
        scenario_name=scenario_name,
        solver_name='pfi',
        value=v_pfi,
        policy_k=pk_pfi,
        policy_b=pb_pfi,
        bond_price=q_pfi,
        params=params,
        shock_params=None,
        grid_config=grid_config_risky,
        metrics={
            'value_vfi_gap_max_abs': value_gap,
            'policy_k_vfi_gap_max_abs': k_gap,
            'policy_b_vfi_gap_max_abs': b_gap,
            'q_vfi_gap_max_abs': q_gap,
            'dataset_fingerprint': train_bundle.fingerprint,
        },
        overwrite=True,
        verbose=False,
    )

    print(f'  max gaps: V={value_gap:.3e}, k={k_gap:.3e}, b={b_gap:.3e}, q={q_gap:.3e}')


In [ ]:
# =============================================================================
# 8. Load Available NN Checkpoints (for comparison)
# =============================================================================
nn_basic = {}
for method in ['lr', 'er', 'br']:
    ckpt = NN_CHECKPOINTS_DIR / 'basic' / 'baseline' / method
    if ckpt.exists():
        nn_basic[method] = load_training_result(str(ckpt), load_target_nets=False, verbose=False)

nn_risky = None
nn_risky_ckpt = NN_CHECKPOINTS_DIR / 'risky' / 'baseline'
if nn_risky_ckpt.exists():
    nn_risky = load_training_result(str(nn_risky_ckpt), load_target_nets=False, verbose=False)

print('Loaded NN basic methods:', sorted(nn_basic.keys()))
print('Loaded NN risky BR:', nn_risky is not None)


In [ ]:
# =============================================================================
# 9. Basic Baseline Comparison: DDP (VFI/PFI) vs NN (LR/ER/BR)
# =============================================================================
basic_base = ddp_basic_results['baseline']
model_basic = basic_base['model']
k_vals = model_basic.k_grid.numpy()
z_vals = model_basic.z_grid.numpy()
z_idx = len(z_vals) // 2
z_fixed = float(z_vals[z_idx])

policy_vfi = basic_base['vfi']['policy_k'].numpy()[z_idx, :]
policy_pfi = basic_base['pfi']['policy_k'].numpy()[z_idx, :]

plt.figure(figsize=(10, 6))
plt.plot(k_vals, policy_vfi, label='DDP VFI', linewidth=2.5)
plt.plot(k_vals, policy_pfi, '--', label='DDP PFI', linewidth=2.5)

nn_rmse = {}
for method, result in nn_basic.items():
    net = result['_policy_net']
    k_in = tf.constant(k_vals.reshape(-1, 1), dtype=tf.float32)
    z_in = tf.constant(np.full((len(k_vals), 1), z_fixed), dtype=tf.float32)
    k_next = net(k_in, z_in).numpy().reshape(-1)
    nn_rmse[method] = float(np.sqrt(np.mean((k_next - policy_pfi) ** 2)))
    plt.plot(k_vals, k_next, label=f'NN {method.upper()}')

plt.plot(k_vals, k_vals, ':', color='gray', label='45-degree')
plt.title(f'Basic Baseline Policy Comparison at median z={z_fixed:.3f}')
plt.xlabel('Current capital k')
plt.ylabel("Next capital k'")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
basic_fig_path = DDP_FIGURES_DIR / 'basic_baseline_policy_compare.png'
plt.savefig(basic_fig_path, dpi=200, bbox_inches='tight')
plt.show()

print('DDP VFI-PFI max policy gap:', basic_base['gaps']['policy_k_max_abs'])
for method in sorted(nn_rmse):
    print(f'RMSE(NN {method.upper()} vs DDP PFI) = {nn_rmse[method]:.6f}')
print(f'Saved figure: {basic_fig_path}')


In [ ]:
# =============================================================================
# 10. Risky Baseline Comparison: DDP (VFI/PFI) vs NN BR
# =============================================================================
risky_base = ddp_risky_results['baseline']
model_risky = risky_base['model']

z_vals = model_risky.z_grid.numpy()
k_vals = model_risky.k_grid.numpy()
b_vals = model_risky.b_grid.numpy()

z_idx = len(z_vals) // 2
k_idx = len(k_vals) // 2
z_fixed = float(z_vals[z_idx])
k_fixed = float(k_vals[k_idx])

b_policy_vfi = risky_base['vfi']['policy_b'].numpy()[z_idx, k_idx, :]
b_policy_pfi = risky_base['pfi']['policy_b'].numpy()[z_idx, k_idx, :]
k_policy_vfi = risky_base['vfi']['policy_k'].numpy()[z_idx, k_idx, :]
k_policy_pfi = risky_base['pfi']['policy_k'].numpy()[z_idx, k_idx, :]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax_b, ax_k = axes

ax_b.plot(b_vals, b_policy_vfi, label='DDP VFI', linewidth=2.5)
ax_b.plot(b_vals, b_policy_pfi, '--', label='DDP PFI', linewidth=2.5)
ax_b.plot(b_vals, b_vals, ':', color='gray', label='45-degree')

ax_k.plot(b_vals, k_policy_vfi, label='DDP VFI', linewidth=2.5)
ax_k.plot(b_vals, k_policy_pfi, '--', label='DDP PFI', linewidth=2.5)

if nn_risky is not None and '_policy_net' in nn_risky:
    net = nn_risky['_policy_net']
    k_in = tf.constant(np.full((len(b_vals), 1), k_fixed), dtype=tf.float32)
    b_in = tf.constant(b_vals.reshape(-1, 1), dtype=tf.float32)
    z_in = tf.constant(np.full((len(b_vals), 1), z_fixed), dtype=tf.float32)
    k_next_nn, b_next_nn = net(k_in, b_in, z_in)
    k_next_nn = k_next_nn.numpy().reshape(-1)
    b_next_nn = b_next_nn.numpy().reshape(-1)

    ax_b.plot(b_vals, b_next_nn, label='NN BR')
    ax_k.plot(b_vals, k_next_nn, label='NN BR')

    rmse_b = float(np.sqrt(np.mean((b_next_nn - b_policy_pfi) ** 2)))
    rmse_k = float(np.sqrt(np.mean((k_next_nn - k_policy_pfi) ** 2)))
    print(f"RMSE(NN BR vs DDP PFI): b'={rmse_b:.6f}, k'={rmse_k:.6f}")

ax_b.set_title(f'Risky baseline debt policy at z={z_fixed:.3f}, k={k_fixed:.2f}')
ax_b.set_xlabel('Current debt b')
ax_b.set_ylabel("Next debt b'")
ax_b.grid(alpha=0.3)
ax_b.legend()

ax_k.set_title(f'Risky baseline capital policy at z={z_fixed:.3f}, k={k_fixed:.2f}')
ax_k.set_xlabel('Current debt b')
ax_k.set_ylabel("Next capital k'")
ax_k.grid(alpha=0.3)
ax_k.legend()

plt.tight_layout()
risky_fig_path = DDP_FIGURES_DIR / 'risky_baseline_policy_compare.png'
plt.savefig(risky_fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('DDP risky VFI-PFI max gaps:', risky_base['gaps'])
print(f'Saved figure: {risky_fig_path}')


In [ ]:
# =============================================================================
# 11. Save DDP Run Metadata
# =============================================================================
ddp_meta = {
    'ddp_run_name': DDP_RUN_NAME,
    'nn_reference_checkpoints': str(NN_CHECKPOINTS_DIR),
    'dataset_path': str(DATASET_PATH),
    'dataset_fingerprint': train_bundle.fingerprint,
    'bounds': {'k': list(k_bounds), 'log_z': list(logz_bounds), 'b': list(b_bounds)},
    'grid_config_basic': dataclasses.asdict(grid_config_basic),
    'grid_config_risky': dataclasses.asdict(grid_config_risky),
    'basic_scenarios': list(BASIC_SCENARIOS.keys()),
    'risky_scenarios': list(RISKY_SCENARIOS.keys()),
}

meta_path = DDP_CHECKPOINTS_DIR / 'ddp_run_meta.json'
with open(meta_path, 'w') as f:
    json.dump(ddp_meta, f, indent=2)

print(f'DDP metadata saved: {meta_path}')
print('Notebook complete.')
